# Data Preparation

In [ ]:
# --- CELL 1: Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/ucf101_pipeline", exist_ok=True)


In [ ]:
# --- CELL 2: Get UCF101.rar - download once, reuse forever after ---
DRIVE_RAR = "/content/drive/MyDrive/ucf101_pipeline/UCF101.rar"
LOCAL_RAR = "/content/UCF101.rar"

if os.path.exists(DRIVE_RAR):
    print("Found UCF101.rar already saved in Drive - copying to local disk (fast)...")
    !cp "{DRIVE_RAR}" "{LOCAL_RAR}"
else:
    print("Not found in Drive yet - downloading from UCF servers (slow, ~6.9GB, one-time only)...")
    !wget --no-check-certificate https://www.crcv.ucf.edu/data/UCF101/UCF101.rar -O "{LOCAL_RAR}"
    print("Backing up to Drive so this NEVER needs downloading again...")
    !cp "{LOCAL_RAR}" "{DRIVE_RAR}"

print("Done - UCF101.rar is ready locally.")

In [ ]:
# --- CELL 3: Unpack the archive (local disk - much faster than unpacking on Drive) ---
!apt-get install -y unrar
!unrar x -y "{LOCAL_RAR}"
# Creates UCF-101/ locally, containing 101 class subfolders

In [ ]:
# --- CELL 4: Copy out only the 10 needed classes (fast, local-to-local) ---
import shutil

SELECTED_CLASSES = [
    "BenchPress", "Basketball", "Biking", "Fencing", "GolfSwing",
    "HorseRiding", "Kayaking", "Skiing", "SoccerJuggling", "TennisSwing",
]

SOURCE_DIR = "UCF-101"
SELECTED_DIR = "selected_classes"
os.makedirs(SELECTED_DIR, exist_ok=True)

for class_name in SELECTED_CLASSES:
    src = os.path.join(SOURCE_DIR, class_name)
    dst = os.path.join(SELECTED_DIR, class_name)
    if not os.path.exists(src):
        print(f"WARNING: folder not found for class '{class_name}'")
        continue
    if os.path.exists(dst):
        print(f"{class_name}: already copied - skipping")
        continue
    shutil.copytree(src, dst)
    print(f"{class_name}: {len(os.listdir(dst))} video clips copied")

In [ ]:
# --- CELL 5: Extract frames at 1fps, PER CLASS, backing up to Drive after each class ---
import cv2

FRAMES_DIR_LOCAL = "extracted_frames"
FRAMES_DIR_DRIVE = "/content/drive/MyDrive/ucf101_pipeline/extracted_frames"
os.makedirs(FRAMES_DIR_LOCAL, exist_ok=True)
os.makedirs(FRAMES_DIR_DRIVE, exist_ok=True)

for class_name in SELECTED_CLASSES:
    drive_class_dir = os.path.join(FRAMES_DIR_DRIVE, class_name)

    # --- RESUME CHECK: skip classes already finished and backed up in a previous session ---
    if os.path.exists(drive_class_dir) and len(os.listdir(drive_class_dir)) > 0:
        print(f"{class_name}: already extracted in a previous session "
              f"({len(os.listdir(drive_class_dir))} frames in Drive) - skipping")
        continue

    class_in = os.path.join(SELECTED_DIR, class_name)
    class_out_local = os.path.join(FRAMES_DIR_LOCAL, class_name)
    os.makedirs(class_out_local, exist_ok=True)
    if not os.path.exists(class_in):
        continue

    saved_idx = 0
    for video_file in os.listdir(class_in):
        video_path = os.path.join(class_in, video_file)
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not fps or fps <= 0:
            fps = 25
        frame_interval = int(round(fps))

        frame_idx, video_saved = 0, 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % frame_interval == 0:
                out_name = f"{os.path.splitext(video_file)[0]}_f{video_saved}.jpg"
                cv2.imwrite(os.path.join(class_out_local, out_name), frame)
                video_saved += 1
                saved_idx += 1
            frame_idx += 1
        cap.release()

    # --- BACK UP THIS CLASS TO DRIVE IMMEDIATELY, before moving to the next class ---
    shutil.copytree(class_out_local, drive_class_dir, dirs_exist_ok=True)
    print(f"{class_name}: {saved_idx} frames extracted AND backed up to Drive.")

print("\nAll classes processed. Frames are safe in Drive at: ucf101_pipeline/extracted_frames")

In [ ]:
# --- CELL 7: Split into train/val/test, 70:15:15, stratified ---
!pip install split-folders
import splitfolders

splitfolders.ratio(
    "/content/drive/MyDrive/ucf101_pipeline/extracted_frames",  # reading straight from Drive
    output="/content/drive/MyDrive/ucf101_pipeline/dataset",     # writing straight to Drive
    seed=42,
    ratio=(0.70, 0.15, 0.15),
)
print("Done. Final dataset saved at: ucf101_pipeline/dataset/{train,val,test}")

In [ ]:
import os
frames_dir = "/content/drive/MyDrive/ucf101_pipeline/extracted_frames"
for class_name in sorted(os.listdir(frames_dir)):
    count = len(os.listdir(os.path.join(frames_dir, class_name)))
    print(f"{class_name}: {count} files")

In [ ]:
import os
dataset_dir = "/content/drive/MyDrive/ucf101_pipeline/dataset"
for split in ["train", "val", "test"]:
    split_path = os.path.join(dataset_dir, split)
    print(f"\n{split.upper()}:")
    total = 0
    for class_name in sorted(os.listdir(split_path)):
        count = len(os.listdir(os.path.join(split_path, class_name)))
        total += count
        print(f"  {class_name}: {count}")
    print(f"  --> {split} total: {total}")